# N6 — FX and Funding

## Decision question

How much committed funding and per-currency hedge protection should the company
buy, given cost, remaining downside and approval authority?

Receivables and payables are evaluated separately. Forward costs are explicit
one-time scenario assumptions, not live executable quotes.


In [ ]:
from pathlib import Path
import json
import sys

# Find the public package locally. A fresh Colab runtime downloads the same
# participant-safe assets from the repository.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for source_candidate in (candidate / 'src', candidate / 'CFOPackV002' / 'src'):
        if (source_candidate / 'workshop_bootstrap.py').exists():
            sys.path.insert(0, str(source_candidate))
            break

try:
    from workshop_bootstrap import bootstrap
except ImportError:
    from urllib.request import urlopen
    bootstrap_url = (
        'https://raw.githubusercontent.com/VinayaSharada/'
        'KateelLearningDemosToStudents/cfopack-v002-v2.0.0-alpha.1/CFOPackV002/src/workshop_bootstrap.py'
    )
    namespace = {}
    exec(compile(urlopen(bootstrap_url).read(), bootstrap_url, 'exec'), namespace)
    bootstrap = namespace['bootstrap']

ROOT, OUTPUT_DIR = bootstrap()
from cfopack_v002 import (
    analyze_fx,
    default_decisions,
    load_inputs,
    load_manifest,
    run_pipeline,
)
import workshop_visuals as viz
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    # Keep the notebooks runnable from a minimal local Python environment as
    # well as Colab/Jupyter. Rich notebook rendering remains the default.
    def Markdown(value):
        return value

    def display(value):
        print(value)

manifest = load_manifest(ROOT / 'config' / 'scenario_manifest.json')
decision_file = OUTPUT_DIR / 'N0_team_decisions.json'
if decision_file.exists():
    DECISIONS = json.loads(decision_file.read_text(encoding='utf-8'))
else:
    DECISIONS = default_decisions(manifest)


In [ ]:
data = load_inputs(ROOT / 'data' / 'synthetic')
viz.data_snapshot(data, OUTPUT_DIR, 'N6')


In [ ]:
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
print(f"Scenario {summary['scenario_version']} calculated for {DECISIONS['team_name']}")


## FX exposure and hedge economics


In [ ]:
fx = pd.read_csv(OUTPUT_DIR / 'N6_fx_decision.csv')
funding = pd.read_csv(OUTPUT_DIR / 'N6_funding_summary.csv')
display(fx)
viz.fx_chart(fx, OUTPUT_DIR)
display(funding)
print(f"Total adverse loss before: ${fx['adverse_loss_before'].sum():,.0f}")
print(f"Total adverse loss after:  ${fx['adverse_loss_after'].sum():,.0f}")
print(f"One-time forward cost:     ${fx['one_time_forward_cost'].sum():,.0f}")


## Team decision

For each currency, defend the proposed hedge ratio and identify the exposure
direction. Then state whether the facility draw preserves enough unused
headroom for an additional shock.


### Before moving on

Record your interpretation in the participant workbook. Do not copy a chart
without also recording the assumption and decision it supports.
